In [1]:
import os
os.chdir("../../..")

In [2]:
import torch
from models.DeepCNN import DeepCNN
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
from torch.utils.data import random_split

In [3]:
class MLROD_dataset(Dataset):
  def __init__(self,path):
    """
    path is a string containing the path to the pkl dataset
    """
    super().__init__()   
    self.y, self.X = pickle.load(open(path, 'rb'))
    self.y = list(self.y) #y is a list with containing the name of the chemical corresponding to X
    self.X = list(self.X) #X is a list with each element of the list containing a 1024 time series data    

    #To remove the Unknown samples from the dataset
    i = 0
    while i<len(self.y):
      if self.y[i]==15:
        self.y.pop(i)
        self.X.pop(i)
      
      else:
        i+=1

  def __len__(self):
    return len(self.y)

  def __getitem__(self,index):
    data = torch.Tensor(self.X[index]) #of shape (1,1024)
    data = (data-data.min())/(data.max()-data.min())
    label = self.y[index]
    return data,label

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
def test_f1_ML(model,test_set):
    X_test = [test_set[i][0].squeeze() for i in range(len(test_set))]
    y_test = [test_set[i][1] for i in range(len(test_set))]

    all_preds = model.predict(X_test)
    all_preds = np.array(all_preds)
    all_targets = np.array(y_test)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    return accuracy, precision, recall, f1

In [7]:
def test_f1_Random(test_set,count,train_set):
    y_test = [test_set[i][1] for i in range(len(test_set))]
    y_train = [train_set[i][1] for i in range(len(train_set))]
    classes = sorted(list(set(y_train)))

    rng = np.random.default_rng(count)
    all_preds = rng.choice(classes,len(y_test))

    all_targets = np.array(y_test)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    return accuracy, precision, recall, f1

In [8]:
def test_f1_majority(test_set,all_train_set,run):
    y_test = [test_set[i][1] for i in range(len(test_set))]

    generator = torch.manual_seed(43+run)
    train_set, _ = random_split(all_train_set,[0.8,0.2],generator=generator)
    y_train = [train_set[i][1] for i in range(len(train_set))]

    count = {}
    max_freq = 0
    max_class = None

    for y in y_train:
        if y not in count:
            count[y] = 1
        
        else:
            count[y] += 1
        
        if count[y]>max_freq:
            max_freq = count[y]
            max_class = y
    

    all_preds = np.array([max_class for _ in range(len(y_test))])

    all_targets = np.array(y_test)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    return accuracy, precision, recall, f1

In [9]:
def test_f1_wrapper(model1,model2,model3,model4,model5,device,test_dataloader,test_dataloader_granite_0,test_dataloader_granite_50,test_dataloader_gabbro_0,test_dataloader_gabbro_50,model_type,train_set=None):

    if model_type=="DL":
        all_acc1, _, _, all_f11 = test_f1(model1,device,test_dataloader)
        gn0_acc1, _, _, gn0_f11 = test_f1(model1,device,test_dataloader_granite_0)
        gn50_acc1, _, _, gn50_f11 = test_f1(model1,device,test_dataloader_granite_50)
        gb0_acc1, _, _, gb0_f11 = test_f1(model1,device,test_dataloader_gabbro_0)
        gb50_acc1, _, _, gb50_f11 = test_f1(model1,device,test_dataloader_gabbro_50)

        all_acc2, _, _, all_f12 = test_f1(model2,device,test_dataloader)
        gn0_acc2, _, _, gn0_f12 = test_f1(model2,device,test_dataloader_granite_0)
        gn50_acc2, _, _, gn50_f12 = test_f1(model2,device,test_dataloader_granite_50)
        gb0_acc2, _, _, gb0_f12 = test_f1(model2,device,test_dataloader_gabbro_0)
        gb50_acc2, _, _, gb50_f12 = test_f1(model2,device,test_dataloader_gabbro_50)

        all_acc3, _, _, all_f13 = test_f1(model3,device,test_dataloader)
        gn0_acc3, _, _, gn0_f13 = test_f1(model3,device,test_dataloader_granite_0)
        gn50_acc3, _, _, gn50_f13 = test_f1(model3,device,test_dataloader_granite_50)
        gb0_acc3, _, _, gb0_f13 = test_f1(model3,device,test_dataloader_gabbro_0)
        gb50_acc3, _, _, gb50_f13 = test_f1(model3,device,test_dataloader_gabbro_50)

        all_acc4, _, _, all_f14 = test_f1(model4,device,test_dataloader)
        gn0_acc4, _, _, gn0_f14 = test_f1(model4,device,test_dataloader_granite_0)
        gn50_acc4, _, _, gn50_f14 = test_f1(model4,device,test_dataloader_granite_50)
        gb0_acc4, _, _, gb0_f14 = test_f1(model4,device,test_dataloader_gabbro_0)
        gb50_acc4, _, _, gb50_f14 = test_f1(model4,device,test_dataloader_gabbro_50)

        all_acc5, _, _, all_f15 = test_f1(model5,device,test_dataloader)
        gn0_acc5, _, _, gn0_f15 = test_f1(model5,device,test_dataloader_granite_0)
        gn50_acc5, _, _, gn50_f15 = test_f1(model5,device,test_dataloader_granite_50)
        gb0_acc5, _, _, gb0_f15 = test_f1(model5,device,test_dataloader_gabbro_0)
        gb50_acc5, _, _, gb50_f15 = test_f1(model5,device,test_dataloader_gabbro_50)

    elif model_type=="RamanNet":
        all_acc1, _, _, all_f11 = test_f1_RamanNet(model1,device,test_dataloader)
        gn0_acc1, _, _, gn0_f11 = test_f1_RamanNet(model1,device,test_dataloader_granite_0)
        gn50_acc1, _, _, gn50_f11 = test_f1_RamanNet(model1,device,test_dataloader_granite_50)
        gb0_acc1, _, _, gb0_f11 = test_f1_RamanNet(model1,device,test_dataloader_gabbro_0)
        gb50_acc1, _, _, gb50_f11 = test_f1_RamanNet(model1,device,test_dataloader_gabbro_50)

        all_acc2, _, _, all_f12 = test_f1_RamanNet(model2,device,test_dataloader)
        gn0_acc2, _, _, gn0_f12 = test_f1_RamanNet(model2,device,test_dataloader_granite_0)
        gn50_acc2, _, _, gn50_f12 = test_f1_RamanNet(model2,device,test_dataloader_granite_50)
        gb0_acc2, _, _, gb0_f12 = test_f1_RamanNet(model2,device,test_dataloader_gabbro_0)
        gb50_acc2, _, _, gb50_f12 = test_f1_RamanNet(model2,device,test_dataloader_gabbro_50)

        all_acc3, _, _, all_f13 = test_f1_RamanNet(model3,device,test_dataloader)
        gn0_acc3, _, _, gn0_f13 = test_f1_RamanNet(model3,device,test_dataloader_granite_0)
        gn50_acc3, _, _, gn50_f13 = test_f1_RamanNet(model3,device,test_dataloader_granite_50)
        gb0_acc3, _, _, gb0_f13 = test_f1_RamanNet(model3,device,test_dataloader_gabbro_0)
        gb50_acc3, _, _, gb50_f13 = test_f1_RamanNet(model3,device,test_dataloader_gabbro_50)

        all_acc4, _, _, all_f14 = test_f1_RamanNet(model4,device,test_dataloader)
        gn0_acc4, _, _, gn0_f14 = test_f1_RamanNet(model4,device,test_dataloader_granite_0)
        gn50_acc4, _, _, gn50_f14 = test_f1_RamanNet(model4,device,test_dataloader_granite_50)
        gb0_acc4, _, _, gb0_f14 = test_f1_RamanNet(model4,device,test_dataloader_gabbro_0)
        gb50_acc4, _, _, gb50_f14 = test_f1_RamanNet(model4,device,test_dataloader_gabbro_50)

        all_acc5, _, _, all_f15 = test_f1_RamanNet(model5,device,test_dataloader)
        gn0_acc5, _, _, gn0_f15 = test_f1_RamanNet(model5,device,test_dataloader_granite_0)
        gn50_acc5, _, _, gn50_f15 = test_f1_RamanNet(model5,device,test_dataloader_granite_50)
        gb0_acc5, _, _, gb0_f15 = test_f1_RamanNet(model5,device,test_dataloader_gabbro_0)
        gb50_acc5, _, _, gb50_f15 = test_f1_RamanNet(model5,device,test_dataloader_gabbro_50)
    
    elif model_type=="ML":
        all_acc1, _, _, all_f11 = test_f1_ML(model1,test_dataloader.dataset)
        gn0_acc1, _, _, gn0_f11 = test_f1_ML(model1,test_dataloader_granite_0.dataset)
        gn50_acc1, _, _, gn50_f11 = test_f1_ML(model1,test_dataloader_granite_50.dataset)
        gb0_acc1, _, _, gb0_f11 = test_f1_ML(model1,test_dataloader_gabbro_0.dataset)
        gb50_acc1, _, _, gb50_f11 = test_f1_ML(model1,test_dataloader_gabbro_50.dataset)

        all_acc2, _, _, all_f12 = test_f1_ML(model2,test_dataloader.dataset)
        gn0_acc2, _, _, gn0_f12 = test_f1_ML(model2,test_dataloader_granite_0.dataset)
        gn50_acc2, _, _, gn50_f12 = test_f1_ML(model2,test_dataloader_granite_50.dataset)
        gb0_acc2, _, _, gb0_f12 = test_f1_ML(model2,test_dataloader_gabbro_0.dataset)
        gb50_acc2, _, _, gb50_f12 = test_f1_ML(model2,test_dataloader_gabbro_50.dataset)

        all_acc3, _, _, all_f13 = test_f1_ML(model3,test_dataloader.dataset)
        gn0_acc3, _, _, gn0_f13 = test_f1_ML(model3,test_dataloader_granite_0.dataset)
        gn50_acc3, _, _, gn50_f13 = test_f1_ML(model3,test_dataloader_granite_50.dataset)
        gb0_acc3, _, _, gb0_f13 = test_f1_ML(model3,test_dataloader_gabbro_0.dataset)
        gb50_acc3, _, _, gb50_f13 = test_f1_ML(model3,test_dataloader_gabbro_50.dataset)

        all_acc4, _, _, all_f14 = test_f1_ML(model4,test_dataloader.dataset)
        gn0_acc4, _, _, gn0_f14 = test_f1_ML(model4,test_dataloader_granite_0.dataset)
        gn50_acc4, _, _, gn50_f14 = test_f1_ML(model4,test_dataloader_granite_50.dataset)
        gb0_acc4, _, _, gb0_f14 = test_f1_ML(model4,test_dataloader_gabbro_0.dataset)
        gb50_acc4, _, _, gb50_f14 = test_f1_ML(model4,test_dataloader_gabbro_50.dataset)

        all_acc5, _, _, all_f15 = test_f1_ML(model5,test_dataloader.dataset)
        gn0_acc5, _, _, gn0_f15 = test_f1_ML(model5,test_dataloader_granite_0.dataset)
        gn50_acc5, _, _, gn50_f15 = test_f1_ML(model5,test_dataloader_granite_50.dataset)
        gb0_acc5, _, _, gb0_f15 = test_f1_ML(model5,test_dataloader_gabbro_0.dataset)
        gb50_acc5, _, _, gb50_f15 = test_f1_ML(model5,test_dataloader_gabbro_50.dataset)  

    elif model_type=="Random":
        all_acc1, _, _, all_f11 = test_f1_Random(test_dataloader.dataset,1,train_set)
        gn0_acc1, _, _, gn0_f11 = test_f1_Random(test_dataloader_granite_0.dataset,1,train_set)
        gn50_acc1, _, _, gn50_f11 = test_f1_Random(test_dataloader_granite_50.dataset,1,train_set)
        gb0_acc1, _, _, gb0_f11 = test_f1_Random(test_dataloader_gabbro_0.dataset,1,train_set)
        gb50_acc1, _, _, gb50_f11 = test_f1_Random(test_dataloader_gabbro_50.dataset,1,train_set)

        all_acc2, _, _, all_f12 = test_f1_Random(test_dataloader.dataset,2,train_set)
        gn0_acc2, _, _, gn0_f12 = test_f1_Random(test_dataloader_granite_0.dataset,2,train_set)
        gn50_acc2, _, _, gn50_f12 = test_f1_Random(test_dataloader_granite_50.dataset,2,train_set)
        gb0_acc2, _, _, gb0_f12 = test_f1_Random(test_dataloader_gabbro_0.dataset,2,train_set)
        gb50_acc2, _, _, gb50_f12 = test_f1_Random(test_dataloader_gabbro_50.dataset,2,train_set)

        all_acc3, _, _, all_f13 = test_f1_Random(test_dataloader.dataset,3,train_set)
        gn0_acc3, _, _, gn0_f13 = test_f1_Random(test_dataloader_granite_0.dataset,3,train_set)
        gn50_acc3, _, _, gn50_f13 = test_f1_Random(test_dataloader_granite_50.dataset,3,train_set)
        gb0_acc3, _, _, gb0_f13 = test_f1_Random(test_dataloader_gabbro_0.dataset,3,train_set)
        gb50_acc3, _, _, gb50_f13 = test_f1_Random(test_dataloader_gabbro_50.dataset,3,train_set)

        all_acc4, _, _, all_f14 = test_f1_Random(test_dataloader.dataset,4,train_set)
        gn0_acc4, _, _, gn0_f14 = test_f1_Random(test_dataloader_granite_0.dataset,4,train_set)
        gn50_acc4, _, _, gn50_f14 = test_f1_Random(test_dataloader_granite_50.dataset,4,train_set)
        gb0_acc4, _, _, gb0_f14 = test_f1_Random(test_dataloader_gabbro_0.dataset,4,train_set)
        gb50_acc4, _, _, gb50_f14 = test_f1_Random(test_dataloader_gabbro_50.dataset,4,train_set)

        all_acc5, _, _, all_f15 = test_f1_Random(test_dataloader.dataset,5,train_set)
        gn0_acc5, _, _, gn0_f15 = test_f1_Random(test_dataloader_granite_0.dataset,5,train_set)
        gn50_acc5, _, _, gn50_f15 = test_f1_Random(test_dataloader_granite_50.dataset,5,train_set)
        gb0_acc5, _, _, gb0_f15 = test_f1_Random(test_dataloader_gabbro_0.dataset,5,train_set)
        gb50_acc5, _, _, gb50_f15 = test_f1_Random(test_dataloader_gabbro_50.dataset,5,train_set)  

    elif model_type=="Majority":
        all_acc1, _, _, all_f11 = test_f1_majority(test_dataloader.dataset,train_set,0)
        gn0_acc1, _, _, gn0_f11 = test_f1_majority(test_dataloader_granite_0.dataset,train_set,0)
        gn50_acc1, _, _, gn50_f11 = test_f1_majority(test_dataloader_granite_50.dataset,train_set,0)
        gb0_acc1, _, _, gb0_f11 = test_f1_majority(test_dataloader_gabbro_0.dataset,train_set,0)
        gb50_acc1, _, _, gb50_f11 = test_f1_majority(test_dataloader_gabbro_50.dataset,train_set,0)

        all_acc2, _, _, all_f12 = test_f1_majority(test_dataloader.dataset,train_set,1)
        gn0_acc2, _, _, gn0_f12 = test_f1_majority(test_dataloader_granite_0.dataset,train_set,1)
        gn50_acc2, _, _, gn50_f12 = test_f1_majority(test_dataloader_granite_50.dataset,train_set,1)
        gb0_acc2, _, _, gb0_f12 = test_f1_majority(test_dataloader_gabbro_0.dataset,train_set,1)
        gb50_acc2, _, _, gb50_f12 = test_f1_majority(test_dataloader_gabbro_50.dataset,train_set,1)

        all_acc3, _, _, all_f13 = test_f1_majority(test_dataloader.dataset,train_set,2)
        gn0_acc3, _, _, gn0_f13 = test_f1_majority(test_dataloader_granite_0.dataset,train_set,2)
        gn50_acc3, _, _, gn50_f13 = test_f1_majority(test_dataloader_granite_50.dataset,train_set,2)
        gb0_acc3, _, _, gb0_f13 = test_f1_majority(test_dataloader_gabbro_0.dataset,train_set,2)
        gb50_acc3, _, _, gb50_f13 = test_f1_majority(test_dataloader_gabbro_50.dataset,train_set,2)

        all_acc4, _, _, all_f14 = test_f1_majority(test_dataloader.dataset,train_set,3)
        gn0_acc4, _, _, gn0_f14 = test_f1_majority(test_dataloader_granite_0.dataset,train_set,3)
        gn50_acc4, _, _, gn50_f14 = test_f1_majority(test_dataloader_granite_50.dataset,train_set,3)
        gb0_acc4, _, _, gb0_f14 = test_f1_majority(test_dataloader_gabbro_0.dataset,train_set,3)
        gb50_acc4, _, _, gb50_f14 = test_f1_majority(test_dataloader_gabbro_50.dataset,train_set,3)

        all_acc5, _, _, all_f15 = test_f1_majority(test_dataloader.dataset,train_set,4)
        gn0_acc5, _, _, gn0_f15 = test_f1_majority(test_dataloader_granite_0.dataset,train_set,4)
        gn50_acc5, _, _, gn50_f15 = test_f1_majority(test_dataloader_granite_50.dataset,train_set,4)
        gb0_acc5, _, _, gb0_f15 = test_f1_majority(test_dataloader_gabbro_0.dataset,train_set,4)
        gb50_acc5, _, _, gb50_f15 = test_f1_majority(test_dataloader_gabbro_50.dataset,train_set,4)  

    all_acc = np.array([all_acc1,all_acc2,all_acc3,all_acc4,all_acc5])
    all_f1 = np.array([all_f11,all_f12,all_f13,all_f14,all_f15])

    gn0_acc = np.array([gn0_acc1,gn0_acc2,gn0_acc3,gn0_acc4,gn0_acc5])
    gn0_f1 = np.array([gn0_f11,gn0_f12,gn0_f13,gn0_f14,gn0_f15])

    gn50_acc = np.array([gn50_acc1,gn50_acc2,gn50_acc3,gn50_acc4,gn50_acc5])
    gn50_f1 = np.array([gn50_f11,gn50_f12,gn50_f13,gn50_f14,gn50_f15])

    gb0_acc = np.array([gb0_acc1,gb0_acc2,gb0_acc3,gb0_acc4,gb0_acc5])
    gb0_f1 = np.array([gb0_f11,gb0_f12,gb0_f13,gb0_f14,gb0_f15])

    gb50_acc = np.array([gb50_acc1,gb50_acc2,gb50_acc3,gb50_acc4,gb50_acc5])
    gb50_f1 = np.array([gb50_f11,gb50_f12,gb50_f13,gb50_f14,gb50_f15])

    print(f"{round(gn0_acc.mean(),2)}\\% (\u00B1 {round(gn0_acc.std(),2)}) & {round(gn0_f1.mean(),4)} (\u00B1 {round(gn0_f1.std(),4)}) & {round(gn50_acc.mean(),2)}\\% (\u00B1 {round(gn50_acc.std(),2)}) & {round(gn50_f1.mean(),4)} (\u00B1 {round(gn50_f1.std(),4)}) & {round(gb0_acc.mean(),2)}\\% (\u00B1 {round(gb0_acc.std(),2)}) & {round(gb0_f1.mean(),4)} (\u00B1 {round(gb0_f1.std(),4)}) & {round(gb50_acc.mean(),2)}\\% (\u00B1 {round(gb50_acc.std(),2)}) & {round(gb50_f1.mean(),4)} (\u00B1 {round(gb50_f1.std(),4)}) & {round(all_acc.mean(),2)}\\% (\u00B1 {round(all_acc.std(),2)}) & {round(all_f1.mean(),4)} (\u00B1 {round(all_f1.std(),4)}) \\\\")

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [11]:
train_set = MLROD_dataset("datasets/MLROD/MLROD_train.pkl")

test_set = MLROD_dataset("datasets/MLROD/MLROD_test.pkl")
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

test_set_granite_0 = MLROD_dataset("datasets/MLROD/MLROD_test_granite_0.pkl")
test_set_granite_50 = MLROD_dataset("datasets/MLROD/MLROD_test_granite_50.pkl")
test_set_gabbro_0 = MLROD_dataset("datasets/MLROD/MLROD_test_gabbro_0.pkl")
test_set_gabbro_50 = MLROD_dataset("datasets/MLROD/MLROD_test_gabbro_50.pkl")

test_loader_granite_0 = DataLoader(test_set_granite_0, batch_size=32, num_workers=8,shuffle=False)
test_loader_granite_50 = DataLoader(test_set_granite_50, batch_size=32, num_workers=8,shuffle=False)
test_loader_gabbro_0 = DataLoader(test_set_gabbro_0, batch_size=32, num_workers=8,shuffle=False)
test_loader_gabbro_50 = DataLoader(test_set_gabbro_50, batch_size=32, num_workers=8,shuffle=False)

print(f"The number of elements in the test set is {len(test_loader_granite_0.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_granite_50.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_gabbro_0.dataset)}")
print(f"The number of elements in the test set is {len(test_loader_gabbro_50.dataset)}")

The number of elements in the test set is 34903
The number of elements in the test set is 11028
The number of elements in the test set is 5183
The number of elements in the test set is 8952
The number of elements in the test set is 9740


In [12]:
deepcnn1 = DeepCNN().to(device)
deepcnn1.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_deepcnn_0_6_99.98_.pt",weights_only=True))

deepcnn2 = DeepCNN().to(device)
deepcnn2.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_deepcnn_1_5_99.99_.pt",weights_only=True))

deepcnn3 = DeepCNN().to(device)
deepcnn3.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_deepcnn_2_5_99.98_.pt",weights_only=True))

deepcnn4 = DeepCNN().to(device)
deepcnn4.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_deepcnn_3_1_99.97_.pt",weights_only=True))

deepcnn5 = DeepCNN().to(device)
deepcnn5.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_deepcnn_4_3_99.99_.pt",weights_only=True))

RamanFormer1 = RamanFormer().to(device)
RamanFormer1.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanFormer_0_17_99.83_.pt",weights_only=True))

RamanFormer2 = RamanFormer().to(device)
RamanFormer2.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanFormer_1_28_99.91_.pt",weights_only=True))

RamanFormer3 = RamanFormer().to(device)
RamanFormer3.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanFormer_2_27_99.87_.pt",weights_only=True))

RamanFormer4 = RamanFormer().to(device)
RamanFormer4.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanFormer_3_29_99.89_.pt",weights_only=True))

RamanFormer5 = RamanFormer().to(device)
RamanFormer5.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanFormer_4_15_99.87_.pt",weights_only=True))

RamanNet1 = RamanNet().to(device)
RamanNet1.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanNet_0_11_99.68_.pt",weights_only=True))

RamanNet2 = RamanNet().to(device)
RamanNet2.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanNet_1_18_99.75_.pt",weights_only=True))

RamanNet3 = RamanNet().to(device)
RamanNet3.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanNet_2_20_99.8_.pt",weights_only=True))

RamanNet4 = RamanNet().to(device)
RamanNet4.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanNet_3_38_99.74_.pt",weights_only=True))

RamanNet5 = RamanNet().to(device)
RamanNet5.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_RamanNet_4_36_99.83_.pt",weights_only=True))

SANet1 = ScaleAdaptiveNet(num_classes=15).to(device)
SANet1.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_SANet_0_8_99.98_.pt",weights_only=True))

SANet2 = ScaleAdaptiveNet(num_classes=15).to(device)
SANet2.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_SANet_1_6_100.0_.pt",weights_only=True))

SANet3 = ScaleAdaptiveNet(num_classes=15).to(device)
SANet3.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_SANet_2_4_99.99_.pt",weights_only=True))

SANet4 = ScaleAdaptiveNet(num_classes=15).to(device)
SANet4.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_SANet_3_7_99.99_.pt",weights_only=True))

SANet5 = ScaleAdaptiveNet(num_classes=15).to(device)
SANet5.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_SANet_4_11_100.0_.pt",weights_only=True))

transformer1 = ViT(patch_size=128,p=0.1).to(device)
transformer1.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_transformer_0_35_99.62_.pt",weights_only=True))

transformer2 = ViT(patch_size=128,p=0.1).to(device)
transformer2.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_transformer_1_20_99.53_.pt",weights_only=True))

transformer3 = ViT(patch_size=128,p=0.1).to(device)
transformer3.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_transformer_2_32_99.65_.pt",weights_only=True))

transformer4 = ViT(patch_size=128,p=0.1).to(device)
transformer4.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_transformer_3_28_99.63_.pt",weights_only=True))

transformer5 = ViT(patch_size=128,p=0.1).to(device)
transformer5.load_state_dict(torch.load("results/final_multi_run/trained_models/MLROD_transformer_4_17_99.65_.pt",weights_only=True))

with open('results/final_multi_run/trained_models/MLROD_RandomForest_0_99.87_.pkl', 'rb') as file:
    randomforest1 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_RandomForest_1_99.85_.pkl', 'rb') as file:
    randomforest2 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_RandomForest_2_99.91_.pkl', 'rb') as file:
    randomforest3 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_RandomForest_3_99.9_.pkl', 'rb') as file:
    randomforest4 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_RandomForest_4_99.88_.pkl', 'rb') as file:
    randomforest5 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_SVC_0_99.37_.pkl', 'rb') as file:
    svc1 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_SVC_1_99.32_.pkl', 'rb') as file:
    svc2 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_SVC_2_99.36_.pkl', 'rb') as file:
    svc3 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_SVC_3_99.33_.pkl', 'rb') as file:
    svc4 = pickle.load(file)

with open('results/final_multi_run/trained_models/MLROD_SVC_4_99.45_.pkl', 'rb') as file:
    svc5 = pickle.load(file)

In [13]:
test_f1_wrapper(deepcnn1,deepcnn2,deepcnn3,deepcnn4,deepcnn5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="DL")

100%|██████████| 305/305 [00:01<00:00, 279.93it/s]

97.66\% (± 2.18) & 0.7284 (± 0.0248) & 80.76\% (± 1.83) & 0.3779 (± 0.0145) & 97.77\% (± 2.93) & 0.6661 (± 0.0301) & 42.26\% (± 1.71) & 0.1812 (± 0.0095) & 79.72\% (± 1.15) & 0.7132 (± 0.0202) \\


In [14]:
test_f1_wrapper(RamanFormer1,RamanFormer2,RamanFormer3,RamanFormer4,RamanFormer5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="DL")

100%|██████████| 305/305 [00:01<00:00, 225.32it/s]

96.48\% (± 3.12) & 0.5693 (± 0.063) & 77.22\% (± 2.95) & 0.3328 (± 0.0284) & 92.27\% (± 1.68) & 0.4076 (± 0.0486) & 44.39\% (± 3.38) & 0.1455 (± 0.0137) & 78.0\% (± 1.02) & 0.6121 (± 0.0234) \\


In [15]:
test_f1_wrapper(RamanNet1,RamanNet2,RamanNet3,RamanNet4,RamanNet5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="RamanNet")

100%|██████████| 305/305 [00:02<00:00, 133.70it/s]

94.69\% (± 2.5) & 0.5273 (± 0.034) & 80.22\% (± 3.22) & 0.3239 (± 0.0187) & 96.37\% (± 2.64) & 0.5508 (± 0.0352) & 42.12\% (± 1.21) & 0.1629 (± 0.0098) & 78.3\% (± 1.32) & 0.6808 (± 0.0193) \\


In [16]:
test_f1_wrapper(SANet1,SANet2,SANet3,SANet4,SANet5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="DL")

100%|██████████| 305/305 [00:02<00:00, 134.15it/s]

92.3\% (± 1.19) & 0.5543 (± 0.0621) & 84.92\% (± 2.21) & 0.3177 (± 0.0211) & 97.21\% (± 1.79) & 0.715 (± 0.1056) & 48.87\% (± 2.4) & 0.1494 (± 0.0072) & 80.34\% (± 0.88) & 0.6383 (± 0.0231) \\


In [17]:
test_f1_wrapper(transformer1,transformer2,transformer3,transformer4,transformer5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="DL")

100%|██████████| 305/305 [00:02<00:00, 116.08it/s]

95.24\% (± 1.21) & 0.5041 (± 0.061) & 65.7\% (± 1.78) & 0.2953 (± 0.024) & 81.82\% (± 6.41) & 0.3763 (± 0.0288) & 42.61\% (± 3.73) & 0.1506 (± 0.0115) & 72.72\% (± 2.41) & 0.5794 (± 0.0394) \\


In [18]:
test_f1_wrapper(randomforest1,randomforest2,randomforest3,randomforest4,randomforest5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="ML")

92.16\% (± 0.19) & 0.4795 (± 0.0193) & 66.84\% (± 0.62) & 0.2785 (± 0.0102) & 72.21\% (± 2.81) & 0.4114 (± 0.0319) & 41.82\% (± 0.42) & 0.1547 (± 0.0015) & 69.23\% (± 0.78) & 0.5537 (± 0.0122) \\


In [19]:
test_f1_wrapper(svc1,svc2,svc3,svc4,svc5,device,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="ML")

In [ ]:
test_f1_wrapper(None,None,None,None,None,None,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="Random",train_set=train_set)

14.39\% (± 0.32) & 0.103 (± 0.0022) & 11.18\% (± 0.27) & 0.066 (± 0.001) & 12.53\% (± 0.42) & 0.0742 (± 0.002) & 24.8\% (± 0.42) & 0.2193 (± 0.0038) & 6.66\% (± 0.19) & 0.0438 (± 0.0009) \\


In [ ]:
test_f1_wrapper(None,None,None,None,None,None,test_loader,test_loader_granite_0,test_loader_granite_50,test_loader_gabbro_0,test_loader_gabbro_50,model_type="Majority",train_set=train_set)

53.96\% (± 0.0) & 0.1001 (± 0.0) & 38.01\% (± 0.0) & 0.0612 (± 0.0) & 0.0\% (± 0.0) & 0.0 (± 0.0) & 0.0\% (± 0.0) & 0.0 (± 0.0) & 22.69\% (± 0.0) & 0.0247 (± 0.0) \\
